# 4.7 RNA and Protein Together

**Omics series, class 7 of 7.** In 4.5 we measured **RNA** in blood; in 4.6, **proteins** in blood. Many *All of Us* participants have **both**. Today we put them side by side, for the same people:

1. For each gene, does a person's **RNA** level go with their **protein** level?
2. Do RNA and protein tell the **same story** about type 2 diabetes?

Then we pull together what the whole series (4.1–4.7) showed about type 2 diabetes.

This notebook reuses the steps from 4.5 and 4.6, so it explains them briefly; look back at those notebooks for the details.

### By the end of this notebook you'll be able to

1. Explain why RNA and protein levels often **don't** agree, especially in blood
2. Load RNA and protein data for the **same** people, and match each protein to its gene
3. Calculate and interpret RNA–protein **correlations**
4. Compare differential results from two omics layers
5. Summarise what each layer of data can and can't tell us about a disease

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | the *All of Us* RNA-seq and proteomics files (v9), the Controlled Tier dataset |
| 💾 **Saves** | `rna_protein_pairs.csv`: one row per gene–protein pair |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Who has both, and the cohort | 10 |
| 3. Load RNA and proteins | 20 |
| 4. Match proteins to genes | 10 |
| 5. RNA vs protein, gene by gene | 25 |
| 6. Do the diabetes signals agree? | 25 |
| 7. The whole series | 15 |

**Before you start:**

- This notebook works **on its own**, but it's much easier after **4.5** and **4.6**.
- It installs **limma** and **edgeR** if needed (quick if you ran 4.5).

> 💰 **Cost:** runs on a **standard app**. One cohort query (cents to under $1). Loading takes about **10 minutes**: a few minutes for the RNA counts (4.5) and 3 to 5 for the proteins (4.6), both reading large files from the *All of Us* bucket (pennies). Stop the app when you're done.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### From RNA to protein

A gene's RNA is the **template** for its protein. So you might expect people with more RNA to have more protein. Often they don't, for several reasons:

| Reason | Example |
|---|---|
| **Different places** | Our RNA comes from **blood cells**. Many blood proteins are made **elsewhere** (liver, kidney, fat) and released into the blood. Their RNA in blood cells says nothing about them |
| **Different timing** | Proteins can last hours to weeks; RNA often minutes to hours. The two reflect different time windows |
| **Control after RNA** | Cells control how much protein is made from each RNA, and how fast it's broken down |
| **Different samples** | RNA from **whole blood** (cells included); proteins from **plasma** (the liquid part) |
| **Measurement** | Each technology has its own noise and limits |

So we expect RNA and protein to agree mainly for proteins that **blood cells themselves** make and release.

### Correlation

For each gene–protein pair, we calculate the **correlation** across people: does a person with more of the RNA also tend to have more of the protein?

- **Spearman correlation** (ρ, "rho"): like the usual correlation, but uses **ranks**, so a few extreme values don't dominate. From −1 to 1; **0** = no relationship.
- With 600 people, a correlation of about 0.1 or more is unlikely to be chance alone; we'll still correct for testing thousands of pairs.

### Multi-omics integration

Combining layers (DNA, RNA, protein, health records) is called **multi-omics integration**. Today's version is the simplest: line the layers up gene by gene and compare. More advanced methods combine all the layers in one model.

### Glossary

| Term | Means |
|---|---|
| **Multi-omics** | several molecular layers (DNA, RNA, proteins, …) measured in the same people |
| **Spearman correlation (ρ)** | a rank-based correlation, from −1 to 1 |
| **Concordance** | two layers pointing the same way (e.g. both higher in diabetes) |
| **Ensembl gene ID** | a stable ID for a gene, e.g. `ENSG00000148737` (*TCF7L2*) |
| **UniProt ID** | a stable ID for a protein, e.g. `P09769` |

## Part 2. Who has both, and the cohort

### Setup

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

# Extra packages (first time only: 5-10 minutes; quick if you ran 4.5)
user_lib <- Sys.getenv("R_LIBS_USER")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))
for (pkg in c("data.table", "BiocManager")) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg, lib = user_lib, repos = "https://cloud.r-project.org")
}
for (pkg in c("limma", "edgeR")) {
  if (!requireNamespace(pkg, quietly = TRUE)) BiocManager::install(pkg, lib = user_lib, update = FALSE, ask = FALSE)
}
suppressPackageStartupMessages(library(edgeR))   # also loads limma

RNASEQ_DIR <- file.path(aou_data_dir("v9"), "multiomics", "rnaseq")
PROT_DIR   <- file.path(aou_data_dir("v9"), "multiomics", "proteomics")
COUNTS_FILE <- file.path(RNASEQ_DIR, "rnaseqc2", "aou_rnaseq_20260413.gene_reads.gct.gz")
NPX_FILE    <- file.path(PROT_DIR, "npx", "batch_normalized", "Olink_10k_aou_v9_reps_removed.tsv.gz")

### Who has both?

Each data type has a **manifest** listing its participants. `intersect()` finds the IDs in **both** lists.

In [ ]:
rna_manifest  <- read_tsv(file.path(RNASEQ_DIR, "manifest.tsv"), col_types = cols(.default = "c"))
rna_metadata  <- read_tsv(file.path(RNASEQ_DIR, "rnaseq_metadata.tsv"), col_types = cols(.default = "c"))
prot_manifest <- read_tsv(file.path(PROT_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

rna_people  <- unique(rna_manifest$research_id)
prot_people <- unique(prot_manifest$research_id)
both_people <- intersect(rna_people, prot_people)

c(rna = length(rna_people), protein = length(prot_people), both = length(both_people))

**🔍 Read the output:** how many participants have RNA-seq, proteomics, and **both**. In our test run, **8,801** of the 8,980 people with RNA-seq also had proteomics: *All of Us* measured both in almost the same people, on purpose, so that the layers can be compared.

### The cohort

The shared type 2 diabetes query, kept to people with **both** data types, aged 18 to 100, sex at birth *Female* or *Male*, with the RNA quality score from 4.5. Then up to 300 cases and 300 controls.

💰 Cents to under $1.

In [ ]:
cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

cohort %>% filter(person_id %in% both_people) %>% count(cohort) %>% hide_small_counts()

N_PER_GROUP <- 300
set.seed(2026)
samples <- cohort %>%
  filter(person_id %in% both_people,
         cohort %in% c("case", "control"),
         between(age, 18, 100),
         sex_at_birth %in% c("Female", "Male")) %>%
  left_join(rna_metadata %>% transmute(person_id = research_id, rqs = as.numeric(rqs)), by = "person_id") %>%
  mutate(group = factor(cohort, levels = c("control", "case"))) %>%
  arrange(person_id) %>%
  group_by(group) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

samples %>% group_by(group) %>%
  summarise(n = n(), mean_age = round(mean(age), 1), pct_female = round(100 * mean(sex_at_birth == "Female"), 1))

**🔍 Read the output:** cases, controls, and excluded among people with both data types; then our sample. These are **different people** from the random samples in 4.5 and 4.6 (they had to have both), so results will differ a little.

## Part 3. Load RNA and proteins

### RNA counts (as in 4.5)

Read only our people's columns from the count file, with `cut`; then filter genes and normalize to **log-CPM** with edgeR. ⏱️ A few minutes.

In [ ]:
file_columns <- str_split(readLines(gzfile(COUNTS_FILE), n = 3)[3], "\t")[[1]]

# Which ID names the sample columns (4.5)? Then find each person's column
id_col <- if (mean(file_columns[-(1:2)] %in% rna_manifest$research_id) > 0.9) "research_id" else "sampleid"
column_lookup <- rna_manifest %>%
  transmute(column_name = .data[[id_col]], research_id) %>%
  filter(research_id %in% samples$person_id, column_name %in% file_columns) %>%
  distinct(research_id, .keep_all = TRUE)
column_numbers <- match(column_lookup$column_name, file_columns)

read_command <- str_glue(
  "gzip -dc {shQuote(COUNTS_FILE)} | tail -n +3 | cut -f 1,2,{str_c(sort(column_numbers), collapse = ',')}"
)
counts_raw <- as.data.frame(data.table::fread(cmd = read_command, sep = "\t", header = TRUE))

genes  <- counts_raw[, c("Name", "Description")]
counts <- as.matrix(counts_raw[, -(1:2)])
rownames(counts) <- genes$Name
colnames(counts) <- column_lookup$research_id[match(colnames(counts), column_lookup$column_name)]
rm(counts_raw)

dim(counts)   # genes x people

### Proteins (as in 4.6)

Keep only our people's rows and the columns we need with `awk`, drop failed measurements, and reshape to a protein × person matrix. ⏱️ 3 to 5 minutes.

In [ ]:
npx_columns <- str_split(system(str_glue("gzip -dc {shQuote(NPX_FILE)} | head -1"), intern = TRUE), "\t")[[1]]

ids_file <- file.path(tempdir(), "ids.txt")
write_lines(samples$person_id, ids_file)

wanted   <- c("ResearchID", "OlinkID", "UniProt", "Assay", "NPX", "SampleQC", "AssayQC")
col_nums <- match(wanted, npx_columns)
stopifnot(!anyNA(col_nums))

awk_command <- str_glue(
  "gzip -dc {shQuote(NPX_FILE)} | ",
  "awk -F'\\t' -v OFS='\\t' -v id={col_nums[1]} ",
  "'NR==FNR {{keep[$1]; next}} FNR==1 || ($id in keep) {{print {str_c('$', col_nums, collapse = ', ')}}}' ",
  "{shQuote(ids_file)} -"
)
npx_long <- data.table::fread(cmd = awk_command, sep = "\t", header = TRUE,
                              colClasses = c(ResearchID = "character", NPX = "numeric")) %>%
  as_tibble() %>%
  mutate(NPX = if_else(SampleQC == "FAIL" | AssayQC == "FAIL", NA_real_, NPX))

protein_info <- npx_long %>% distinct(OlinkID, UniProt, Assay)

npx_wide <- npx_long %>%
  select(OlinkID, ResearchID, NPX) %>%
  pivot_wider(names_from = ResearchID, values_from = NPX, values_fn = mean)
npx <- as.matrix(npx_wide[, -1])
rownames(npx) <- npx_wide$OlinkID
rm(npx_long, npx_wide)

dim(npx)   # proteins x people

### Same people, same order

Keep only people found in **both** matrices, and put both matrices and the sample table in the **same order**. `stopifnot()` stops if anything doesn't line up.

In [ ]:
samples <- samples %>% filter(person_id %in% colnames(counts), person_id %in% colnames(npx))
counts  <- counts[, samples$person_id]
npx     <- npx[, samples$person_id]
stopifnot(identical(colnames(counts), samples$person_id), identical(colnames(npx), samples$person_id))

samples %>% count(group)

# RNA: filter and normalize (4.5)
dge <- DGEList(counts = counts, genes = genes, samples = samples)
dge <- dge[filterByExpr(dge, group = samples$group), , keep.lib.sizes = FALSE]
dge <- calcNormFactors(dge)
logcpm <- cpm(dge, log = TRUE)

# Proteins: keep those measured in at least 90% of people (4.6)
npx <- npx[rowMeans(is.na(npx)) <= 0.10, ]

c(genes = nrow(logcpm), proteins = nrow(npx), people = ncol(npx))

**🔍 Read the output:** the number of people in both, by group; then about 29,000 genes and 5,400 proteins for the same people.

## Part 4. Match proteins to genes

We need to know which gene makes each protein. The **pQTL** file from 4.6 names each protein as **UniProt ID + `_` + Ensembl gene ID** (e.g. `P09769_ENSG00000000938.13`): a ready-made lookup table. Ensembl IDs have a **version** at the end (`.13`), which can differ between files, so we remove it from both sides.

In [ ]:
uniprot_to_gene <- read_tsv(file.path(PROT_DIR, "pqtl", "cis_pqtl", "AoU_v9_COMB.cis_pqtl.txt.gz"),
                            col_select = "phenotype_id", show_col_types = FALSE) %>%
  transmute(UniProt = str_remove(phenotype_id, "_.*"),
            gene_id = str_remove(str_remove(phenotype_id, "^[^_]*_"), "\\..*")) %>%
  distinct()

rna_genes <- tibble(rna_row = rownames(logcpm), gene_id = str_remove(rownames(logcpm), "\\..*"),
                    gene = dge$genes$Description)

pairs <- protein_info %>%
  filter(OlinkID %in% rownames(npx)) %>%
  inner_join(uniprot_to_gene, by = "UniProt") %>%
  inner_join(rna_genes, by = "gene_id") %>%
  distinct(OlinkID, rna_row, .keep_all = TRUE)

nrow(pairs)   # gene-protein pairs with both measurements
head(pairs %>% select(Assay, OlinkID, UniProt, gene, gene_id))

**🔍 Read the output:** how many proteins we could pair with their gene's RNA. Some proteins drop out: their gene was filtered out of the RNA data (too little RNA in blood, Part 3), or they had no pQTL entry. `Assay` (Olink's name) and `gene` (the RNA file's name) should usually agree.

## Part 5. RNA vs protein, gene by gene

For each pair, the **Spearman correlation** across our people. `map2_dbl()` runs `cor()` for each pair of rows: one from `logcpm`, one from `npx`.

`cor.test()` would give a p-value for each; with thousands of pairs, we use a quicker approximation for the p-value from the correlation and the number of people, then adjust for multiple testing with `p.adjust()`.

In [ ]:
pairs <- pairs %>%
  mutate(rho = map2_dbl(rna_row, OlinkID,
                        ~ cor(logcpm[.x, ], npx[.y, ], method = "spearman", use = "pairwise.complete.obs")))

n_people <- ncol(npx)
pairs <- pairs %>%
  mutate(t   = rho * sqrt((n_people - 2) / (1 - rho^2)),
         p   = 2 * pt(-abs(t), df = n_people - 2),
         fdr = p.adjust(p, method = "BH"))

summary(pairs$rho)
pairs %>% summarise(positive_significant = sum(fdr < 0.05 & rho > 0),
                    negative_significant = sum(fdr < 0.05 & rho < 0))

ggplot(pairs, aes(rho)) +
  geom_histogram(bins = 60, fill = "#3182bd") +
  geom_vline(xintercept = 0, linetype = "dashed") +
  theme_bw() +
  labs(x = "Spearman correlation, RNA vs protein (same gene)", y = "Number of gene-protein pairs",
       title = "Do RNA and protein levels go together?")

**🔍 Read the output:**

- Most correlations are **close to 0** (median about 0.01 in our test run): for most genes, a person's blood RNA says little about their plasma protein. That's what Part 1 predicted.
- A **tail** of pairs to the right has clearly positive correlations (about 500 significant in our test run, up to ρ ≈ 0.76). Which proteins are they?

In [ ]:
# The 15 pairs where RNA and protein agree most
pairs %>% arrange(desc(rho)) %>% select(Assay, gene, rho, fdr) %>% mutate(rho = round(rho, 2)) %>% head(15)

**🔍 Read the output:** the best-agreeing pairs are likely proteins that **blood cells make and release**, so the cells in our RNA sample are also the source of the protein. In our test run, most of the top 15 were **immune-cell** proteins:

| Cell type | Proteins (test run) |
|---|---|
| **B cells** | *TCL1A*, *FCRL1*, *FCRL3* |
| **NK cells and cytotoxic T cells** | *GZMH*, *GNLY*, *FCRL6* |
| **Neutrophils** | *OLFM4*, *VSTM1* |

A few top pairs (*MDGA1*, *SPESP1*, *PVALB*) aren't typical blood-cell proteins. One possible explanation: a **genetic variant** that changes both the gene's RNA (an eQTL, 4.5) and its protein (a pQTL, 4.6). People who carry it have more (or less) of both, so the two correlate even if they're made in different places.

Proteins made mainly by the liver, kidney, or fat tissue can't agree: their RNA isn't in blood cells.

**💬 Discuss:** from 4.6, which of the diabetes-related proteins (e.g. *HAVCR1*, *UMOD*, *FABP4*, *LPL*) would you expect to agree with blood RNA? Check below.

### ✍️ Your turn

Look up the correlations for some proteins from 4.6, and some neutrophil genes from 4.5.

<details>
<summary><b>Show answer</b></summary>

```r
pairs %>%
  filter(Assay %in% c("HAVCR1", "UMOD", "FABP4", "LPL", "LTF", "MMP8", "CRISP3", "HP")) %>%
  select(Assay, rho, fdr)
```

Kidney and fat proteins (*HAVCR1*, *UMOD*, *FABP4*, *LPL*) should correlate little or not at all with blood RNA, or be missing (their RNA was filtered out: blood cells barely make it). Neutrophil proteins (*LTF*, *MMP8*, *CRISP3*) have a better chance of correlating, because neutrophils are in our RNA sample. A missing row means the pair wasn't measured in both layers.

</details>

In [ ]:
# Your code here

## Part 6. Do the diabetes signals agree?

Run the two differential analyses from 4.5 and 4.6 on **these** people:

- **RNA:** `voom()` + limma, adjusted for age, sex, and RNA quality (4.5)
- **Protein:** limma on NPX, adjusted for age and sex (4.6)

Then compare each pair's **t-statistic** (the effect divided by its uncertainty: positive = higher in diabetes) across the two layers.

In [ ]:
# RNA (4.5)
design_rna <- model.matrix(~ group + age + sex_at_birth + rqs, data = samples)
fit_rna    <- eBayes(lmFit(voom(dge, design_rna), design_rna))
rna_res    <- topTable(fit_rna, coef = "groupcase", number = Inf) %>% rownames_to_column("rna_row") %>%
  select(rna_row, logFC_rna = logFC, t_rna = t, fdr_rna = adj.P.Val)

# Protein (4.6)
design_prot <- model.matrix(~ group + age + sex_at_birth, data = samples)
fit_prot    <- eBayes(lmFit(npx, design_prot))
prot_res    <- topTable(fit_prot, coef = "groupcase", number = Inf) %>% rownames_to_column("OlinkID") %>%
  select(OlinkID, logFC_prot = logFC, t_prot = t, fdr_prot = adj.P.Val)

pairs <- pairs %>% left_join(rna_res, by = "rna_row") %>% left_join(prot_res, by = "OlinkID")

c(significant_rna = sum(rna_res$fdr_rna < 0.05), significant_protein = sum(prot_res$fdr_prot < 0.05))

# Among pairs: significant in each layer, and in both
pairs %>% count(rna = fdr_rna < 0.05, protein = fdr_prot < 0.05) %>% hide_small_counts()

**🔍 Read the output:** as in 4.5 and 4.6, **many more proteins** than genes differ with diabetes. The table crosses the two: how many pairs are significant in RNA only, protein only, both, or neither (`NA` = a count from 1 to 20, hidden).

In [ ]:
cor(pairs$t_rna, pairs$t_prot, use = "complete.obs")

ggplot(pairs, aes(t_rna, t_prot)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_vline(xintercept = 0, colour = "grey60") +
  geom_point(aes(colour = rho > 0.3), size = 1, alpha = 0.6) +
  geom_text(data = filter(pairs, abs(t_rna) > 4 & abs(t_prot) > 4), aes(label = Assay),
            vjust = -0.7, size = 3, check_overlap = TRUE) +
  scale_colour_manual(values = c(`FALSE` = "grey60", `TRUE` = "#08519c"),
                      labels = c("RNA-protein correlation up to 0.3", "RNA-protein correlation > 0.3"), name = NULL) +
  theme_bw() +
  labs(x = "t-statistic, RNA (case vs control)", y = "t-statistic, protein (case vs control)",
       title = "Diabetes effects: RNA vs protein, same gene")

**🔍 Read the output:**

- Each point is a gene–protein pair. **Top right:** higher in diabetes in **both** layers; **bottom left:** lower in both. Points along the axes changed in **one** layer only.
- The first number is the correlation of the two sets of t-statistics. In our test run it was about **0** (−0.04): across all pairs, a gene's diabetes effect in blood RNA said **nothing** about its diabetes effect in plasma protein.
- The cloud sits **above** the horizontal line: many proteins are a little **higher** in diabetes, whatever their RNA does. One possible reason: the **kidneys** clear many proteins from the blood, and kidney function is often lower in diabetes (remember *HAVCR1* and *UMOD* in 4.6). Lower clearance raises many plasma proteins at once.
- **Dark blue** points are pairs whose RNA and protein go together across people (Part 5). Do they agree more on diabetes, too? They should: if blood cells are the source of the protein, both layers can see the same change.
- Labelled pairs changed strongly in **both** layers.

**💬 Discuss:** a protein is higher in diabetes, but its gene's RNA in blood isn't (e.g. *SERPINF1*, *SAT2* in our test run: protein up, RNA down). What are the possible explanations? (Made in another organ; made at the same rate but cleared more slowly, e.g. by damaged kidneys; a change in how much protein each RNA makes.) How could you test the kidney idea? (Add a measure of kidney function, such as **eGFR** from creatinine in the `measurement` table, to the protein model.)

### Save

In [ ]:
save_data(pairs %>% select(-rna_row), "rna_protein_pairs.csv")

## Part 7. The whole series: what did we learn about type 2 diabetes?

| Class | Layer | Main finding (our test runs) | What it can't tell us |
|---|---|---|---|
| **4.1** DNA intro | DNA | rs7903146's **T** allele more common in cases (OR ≈ 1.17, unadjusted) | anything about cause or tissue |
| **4.2–4.3** GWAS | DNA | adjusted OR ≈ **1.25**; a peak across *TCF7L2*; PLINK and Hail agree | which variant is causal (LD); small sample = low power |
| **4.4** PheWAS | DNA + EHR | the same variant goes with diabetes, its **treatments** and **complications**, and slightly **less** obesity | consequences vs direct effects; collider bias when adjusting |
| **4.5** RNA-seq | RNA (blood) | a few dozen genes differ, many **neutrophil** genes; rs7903146 barely affects *TCF7L2* in **blood** | other tissues (the pancreas); cell-mix vs real change |
| **4.6** Proteomics | protein (plasma) | hundreds of proteins differ, pointing to the **kidney**, **fat**, and **gut**; strong pQTLs | cause vs consequence vs treatment |
| **4.7** RNA + protein | both | RNA and protein agree mainly for proteins **immune cells** make; diabetes effects in the two layers **don't** line up | anything outside blood |

**The big picture:**

- **DNA** tells us about **lifelong risk** (it's fixed at birth), and can help argue about **cause** (Mendelian randomization). But the effects are small, and DNA doesn't show *where* or *when* disease happens.
- **RNA and proteins** show what's happening **now**, including the disease's **consequences** and its **treatment**. But they can't easily separate cause from effect, and blood only shows part of the body.
- **Each layer answers a different question.** Combining them, carefully, is how multi-omics studies get closer to *how* a disease works.

**💬 Discuss:** if you could design one more analysis with *All of Us* data to understand type 2 diabetes better, what would it be, and which layers would it use?

## ⚠️ Common errors

| Message | Means |
|---|---|
| *stopifnot(…) is not TRUE* in Part 3 | The RNA, protein, and sample tables aren't in the same order. Rerun Part 3 from the start |
| Few people in `samples` | Fewer people have both data types in one group. That's fine: the analysis uses whoever is available |
| *cannot allocate vector* | The app ran out of memory with both datasets loaded. Lower `N_PER_GROUP`, or use a bigger app |
| `pairs` has very few rows | The gene IDs didn't match. Check `head(uniprot_to_gene)` and `head(rna_genes)`: both should show IDs like `ENSG00000148737` |

## ✅ Key takeaways

- **RNA and protein** often disagree, especially in blood: many plasma proteins are made in other organs
- Match layers with **stable IDs** (Ensembl, UniProt), not names
- **Correlation across people** shows which genes' RNA and protein go together
- Comparing **differential results** across layers separates signals seen in both from signals unique to one
- Each omics layer answers a **different question**; together they tell a fuller story

**This is the end of the omics series.** Thank you for working through it!

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).